# Demo result reproduction

This notebook allows to reproduce all values from this image:

![demo](https://raw.githubusercontent.com/tensorforger/tensorforger/main/assets/VaeZip_demo.png)


### Step 0: Additional setup

In [1]:
!uv pip install diffusers

Using Python 3.12.14 environment at: /home/user/ML/VaeZip/.venv
Checked 1 package in 1ms


### Step 1: Download FLUX.2 VAE weights

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="black-forest-labs/FLUX.2-klein-4B",
    allow_patterns="vae/*",
    local_dir="models",
)

/home/user/ML/VaeZip/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 886.09it/s]


'/home/user/ML/VaeZip/examples/models'

### Step 2: Load VAE

In [3]:
import torch
from flux_2_vae import Flux2VAE

device = "cuda" if torch.cuda.is_available() else "cpu"

vae = Flux2VAE(path_to_model="models/vae", device=device)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
The config attributes {'force_upcast': True} were passed to AutoencoderKLFlux2, but are not expected and will be ignored. Please verify your config.json configuration file.


### Step 3: Download Image

In [4]:
import os

import requests

os.makedirs("samples", exist_ok=True)

url = "https://raw.githubusercontent.com/tensorforger/tensorforger/main/assets/sample_image.jpg"

response = requests.get(url)
response.raise_for_status()

with open("samples/sample_image.jpg", "wb") as f:
    f.write(response.content)

size = os.path.getsize("samples/sample_image.jpg")
print(f"image size: {size / 1024:.1f} KB")

image size: 201.3 KB


### Step 4: Encode image

In [5]:
import numpy as np
from diffusers.utils import load_image

image = load_image("samples/sample_image.jpg")
image = (
    torch.from_numpy(np.array(image))
    .permute(2, 0, 1)
    .unsqueeze(0)
    .to(device)
    .float()
    .div(255.0)
)

latents = vae.encode(image)
print(latents.shape)

torch.Size([1, 32, 128, 128])


### Step 5: Save latents with two approaches

In [6]:
from safetensors.torch import save_file

import vaezip

# no compression, fp32
save_file({"latents": latents}, "samples/latents.safetensors")

# lbrq_3bit compression
vaezip.save_tensor(latents, "samples/latents.vz.safetensors", codec="lbrq_3bit")

size = os.path.getsize("samples/latents.safetensors")
print(f"latents.safetensors size: {size / 1024 / 1024:.1f} MB")

size = os.path.getsize("samples/latents.vz.safetensors")
print(f"latents.vz.safetensors size: {size / 1024:.1f} KB")

latents.safetensors size: 2.0 MB
latents.vz.safetensors size: 201.1 KB


### Step 6: Reconstruct and measure MSE

In [7]:
from safetensors.torch import load_file

fp32_latents = load_file("samples/latents.safetensors", device=device)["latents"]

lbrq_3bit_latents = vaezip.load_tensor("samples/latents.vz.safetensors", device=device)

fp32_rec_image = vae.decode(fp32_latents)

lbrq_3bit_rec_image = vae.decode(lbrq_3bit_latents)

mse = ((fp32_rec_image - lbrq_3bit_rec_image) ** 2).mean()

print(f"MSE: {mse:.4f}")

MSE: 0.0003


### Step 7 (optional): Visualize images

*Note: the output is not saved in the notebook to reduce size.*

In [ ]:
!uv pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("dark_background")

img1 = fp32_rec_image.cpu().squeeze(0).permute(1, 2, 0).clamp(0, 1)
img2 = lbrq_3bit_rec_image.cpu().squeeze(0).permute(1, 2, 0).clamp(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(24, 24))
axes[0].imshow(img1)
axes[1].imshow(img2)
axes[0].axis("off")
axes[1].axis("off")
axes[0].set_title("fp32")
axes[1].set_title("lbrq_3bit")
plt.show()